# 08A — NCCS source coverage and join integrity audit

Run this notebook **after Notebook 08, cells 2–6**, or independently: it registers the source views and rebuilds the master join for auditing. **Read-only with respect to source Parquet files.** Results are saved under `data/processed/audit/`.

**Important:** 45 files is evidence of expected file *presence*, not proof of complete filings, correct source mappings, or valid modeling labels. Review failures before trusting the models. This audit deliberately does not retrain any model.

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd

base = Path('data/external/nccs/efile_v3_1')
if not base.exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        p = candidate / 'data/external/nccs/efile_v3_1'
        if p.exists():
            base = p
            break
assert base.exists(), f'NCCS source directory not found: {base}'
project = base.parents[3]
out = project / 'data/processed/audit'
out.mkdir(parents=True, exist_ok=True)
print('Source:', base.resolve())
print('Audit outputs:', out.resolve())
con = duckdb.connect(database=':memory:')
con.execute('PRAGMA threads=4')
con.execute("SET memory_limit='8GB'")
for name, tag in [('header','HEADER'), ('summary','SUMMARY'), ('governance','GOVERNANCE')]:
    paths = sorted(base.glob(f'*/*{tag}*.parquet'))
    assert paths, f'No files for {tag}'
    glob_path = (base / '*' / f'*{tag}*.parquet').as_posix().replace("'", "''")
    con.execute(f"CREATE VIEW {name} AS SELECT * FROM read_parquet('{glob_path}', union_by_name=true)")
print('Views registered.')

## A. File inventory, physical Parquet row counts, and expected year coverage

In [ ]:
import pyarrow.parquet as pq
rows = []
for year in range(2010, 2025):
    for table in ['HEADER','SUMMARY','GOVERNANCE']:
        matches = sorted((base / str(year)).glob(f'*{table}*.parquet'))
        rows.append({'year':year,'table':table,'file_count':len(matches),
                     'parquet_rows':sum(pq.ParquetFile(str(p)).metadata.num_rows for p in matches),
                     'bytes':sum(p.stat().st_size for p in matches)})
files_df = pd.DataFrame(rows)
display(files_df.pivot(index='year', columns='table', values='parquet_rows'))
print('Total files:', files_df.file_count.sum())
assert (files_df.file_count == 1).all(), 'Missing or multiple matching files: inspect files_df'
files_df.to_csv(out/'01_file_inventory.csv', index=False)
print('Expected 45 files present. Note: row counts alone cannot prove upstream NCCS completeness.')

## B. Key integrity in each source table

Check NULL `OBJECTID`, duplicates, and whether a single `OBJECTID` maps to more than one EIN/tax year. Duplicate source keys can multiply rows during joins.

In [ ]:
source_audits=[]
for table in ['header','summary','governance']:
    result=con.execute(f"""SELECT COUNT(*) total_rows,
        COUNT(*) FILTER (WHERE OBJECTID IS NULL) null_objectid,
        COUNT(DISTINCT OBJECTID) distinct_objectid
        FROM {table}""").df().iloc[0].to_dict()
    result['table']=table
    result['duplicate_key_rows']=result['total_rows']-result['null_objectid']-result['distinct_objectid']
    source_audits.append(result)
source_df=pd.DataFrame(source_audits)
display(source_df)
source_df.to_csv(out/'02_source_key_integrity.csv',index=False)
for table in ['header','summary','governance']:
    examples=con.execute(f"""SELECT OBJECTID, COUNT(*) AS occurrences
        FROM {table} WHERE OBJECTID IS NOT NULL
        GROUP BY OBJECTID HAVING COUNT(*)>1
        ORDER BY occurrences DESC LIMIT 10""").df()
    print(f'{table} duplicate OBJECTID examples:')
    display(examples)
    examples.to_csv(out/f'02_{table}_duplicate_key_examples.csv',index=False)

## C. Join coverage and row multiplication

Compare exact `OBJECTID` matches against the HEADER population, including one-to-many matches. A left join preserves headers, but can create extra rows when the right-side key is duplicated.

In [ ]:
con.execute("""CREATE TABLE join_audit AS
    SELECT h.OBJECTID, h.EIN2 AS ein,
           TRY_CAST(h.tax_year AS INTEGER) AS tax_year,
           COALESCE(s.n,0) AS summary_matches,
           COALESCE(g.n,0) AS governance_matches
    FROM header h
    LEFT JOIN (SELECT OBJECTID, COUNT(*) n FROM summary GROUP BY OBJECTID) s USING (OBJECTID)
    LEFT JOIN (SELECT OBJECTID, COUNT(*) n FROM governance GROUP BY OBJECTID) g USING (OBJECTID)
""")
join_year=con.execute("""SELECT tax_year, COUNT(*) header_rows,
    SUM(CASE WHEN summary_matches=0 THEN 1 ELSE 0 END) missing_summary,
    SUM(CASE WHEN governance_matches=0 THEN 1 ELSE 0 END) missing_governance,
    SUM(CASE WHEN summary_matches>1 THEN 1 ELSE 0 END) multiple_summary,
    SUM(CASE WHEN governance_matches>1 THEN 1 ELSE 0 END) multiple_governance,
    SUM(GREATEST(summary_matches,1)*GREATEST(governance_matches,1)) expected_join_rows
    FROM join_audit GROUP BY 1 ORDER BY 1""").df()
display(join_year)
join_year.to_csv(out/'03_join_coverage_by_year.csv',index=False)
print('Header rows:',int(join_year.header_rows.sum()))
print('Expected joined rows:',int(join_year.expected_join_rows.sum()))
print('Potential row multiplication:',int((join_year.expected_join_rows-join_year.header_rows).sum()))

## D. Unmatched source rows

These are SUMMARY/GOVERNANCE records without a corresponding HEADER record. They cannot appear in a HEADER-led left join.

In [ ]:
orphans=[]
for table in ['summary','governance']:
    row=con.execute(f"""SELECT COUNT(*) orphan_rows,
       COUNT(DISTINCT s.OBJECTID) orphan_distinct_ids
       FROM {table} s WHERE NOT EXISTS
       (SELECT 1 FROM header h WHERE h.OBJECTID=s.OBJECTID)""").df().iloc[0].to_dict()
    row['table']=table
    orphans.append(row)
orphans_df=pd.DataFrame(orphans)
display(orphans_df)
orphans_df.to_csv(out/'04_unmatched_source_rows.csv',index=False)

## E. EIN/tax-year duplication and the deduplication rule

Notebook 08 selects the latest available filing for each EIN and tax year using the return timestamp, amendment status as a tie-breaker, and OBJECTID for deterministic selection. Identical timestamps and conflicting filings remain a limitation.

In [ ]:
con.execute("""CREATE OR REPLACE TABLE master_audit AS
SELECT h.OBJECTID, TRY_CAST(h.tax_year AS INTEGER) tax_year,
       REGEXP_REPLACE(h.EIN2, '[^0-9]', '', 'g') AS ein,
       TRY_CAST(s.F9_01_NAFB_ASSET_TOT_EOY AS DOUBLE) total_assets,
       s.OBJECTID AS summary_objectid,
       g.OBJECTID AS governance_objectid,
       CAST(g.F9_06_GVRN_ASSET_DIVERSION_X AS VARCHAR) raw_disclosure
FROM header h
LEFT JOIN summary s ON h.OBJECTID=s.OBJECTID
LEFT JOIN governance g ON h.OBJECTID=g.OBJECTID""")
key_audit=con.execute("""SELECT tax_year, COUNT(*) joined_rows,
    COUNT(DISTINCT OBJECTID) distinct_objectids,
    COUNT(DISTINCT ein) distinct_eins,
    COUNT(*) FILTER (WHERE ein IS NULL OR NOT regexp_full_match(ein,'[0-9]{9}')) invalid_eins,
    COUNT(*) FILTER (WHERE tax_year IS NULL OR tax_year NOT BETWEEN 2010 AND 2024) invalid_years
FROM master_audit GROUP BY 1 ORDER BY 1""").df()
display(key_audit)
key_audit.to_csv(out/'05_master_key_quality.csv',index=False)
duplicates=con.execute("""SELECT ein,tax_year,COUNT(*) filings,
    COUNT(DISTINCT OBJECTID) objectids,
    COUNT(DISTINCT UPPER(TRIM(raw_disclosure))) raw_label_variants
FROM master_audit WHERE ein IS NOT NULL AND tax_year IS NOT NULL
GROUP BY 1,2 HAVING COUNT(*)>1 ORDER BY filings DESC LIMIT 100""").df()
display(duplicates.head(20))
duplicates.to_csv(out/'05_duplicate_ein_year_examples.csv',index=False)
print('Duplicate EIN/year groups (first 100 shown):',len(duplicates))

## F. Disclosure label completeness

Notebook 08 preserves missing diversion disclosures as NULL. Only explicitly reported positive and negative disclosures are used for supervised model training and evaluation.

In [ ]:
# Section F: Disclosure label completeness audit

# Raw disclosure values
label_values = con.execute("""
    SELECT
        COALESCE(raw_disclosure, '<NULL>') AS raw_value,
        COUNT(*) AS n
    FROM master_audit
    GROUP BY 1
    ORDER BY n DESC
""").df()

display(label_values.head(40))
label_values.to_csv(out / "06_raw_disclosure_values.csv", index=False)

# Disclosure completeness by tax year
label_year = con.execute("""
    SELECT
        tax_year,
        COUNT(*) AS rows,

        COUNT(*) FILTER (
            WHERE UPPER(TRIM(raw_disclosure)) IN ('1', 'Y', 'YES', 'TRUE')
        ) AS positives,

        COUNT(*) FILTER (
            WHERE UPPER(TRIM(raw_disclosure)) IN ('0', 'N', 'NO', 'FALSE')
        ) AS negatives,

        COUNT(*) FILTER (
            WHERE raw_disclosure IS NULL OR TRIM(raw_disclosure) = ''
        ) AS missing,

        COUNT(*) FILTER (
            WHERE raw_disclosure IS NOT NULL
              AND TRIM(raw_disclosure) <> ''
              AND UPPER(TRIM(raw_disclosure)) NOT IN (
                  '1', 'Y', 'YES', 'TRUE',
                  '0', 'N', 'NO', 'FALSE'
              )
        ) AS unexpected_values

    FROM master_audit
    GROUP BY tax_year
    ORDER BY tax_year
""").df()

display(label_year)
label_year.to_csv(
    out / "06_label_completeness_by_year.csv",
    index=False
)

## G. Financial/governance field completeness by year

This separates missing source rows from rows that joined but contain NULL metrics.

In [ ]:
completeness=con.execute("""SELECT tax_year, COUNT(*) rows,
 COUNT(*) FILTER (WHERE summary_objectid IS NULL) no_summary_join,
 COUNT(*) FILTER (WHERE governance_objectid IS NULL) no_governance_join,
 COUNT(*) FILTER (WHERE total_assets IS NULL) missing_assets,
 COUNT(*) FILTER (WHERE TRY_CAST(raw_disclosure AS VARCHAR) IS NULL) missing_disclosure
 FROM master_audit GROUP BY 1 ORDER BY 1""").df()
display(completeness)
completeness.to_csv(out/'07_field_completeness.csv',index=False)

## H. Summary and audit decision

This audit separates structural integrity from data completeness.

**Structural integrity** evaluates source-file coverage, identifier validity, uniqueness, and join correctness. These checks determine whether the NCCS data can be reliably integrated.

**Data completeness** measures missing SUMMARY and GOVERNANCE records and missing diversion-disclosure labels. Missing values are documented limitations, not automatically evidence of a broken join.

**Duplicate filing consistency** identifies EIN/tax-year combinations with conflicting known disclosure labels or a mixture of known and unknown labels. These cases require additional review because the selected canonical filing may affect the supervised learning target.

**Modeling implications:**
- Preserve unknown disclosure labels as NULL.
- Exclude unknown labels from supervised model training and evaluation.
- Handle missing financial and governance predictors through an explicit preprocessing strategy.
- Review duplicate-label conflicts before finalizing the training dataset.
- Do not interpret self-reported diversion of assets as verified fraud.

A passing structural audit establishes internal consistency, not complete or independently verified NCCS source coverage.

In [ ]:

# H. Final audit summary

# 1. Structural integrity checks
structural_checks = {
    "45 expected source files": bool(
        len(files_df) == 45 and (files_df.file_count == 1).all()
    ),
    "No NULL OBJECTIDs": bool(
        (source_df.null_objectid == 0).all()
    ),
    "No duplicate source OBJECTIDs": bool(
        (source_df.duplicate_key_rows == 0).all()
    ),
    "No join row multiplication": bool(
        (join_year.expected_join_rows == join_year.header_rows).all()
    ),
    "No orphan SUMMARY/GOVERNANCE rows": bool(
        (orphans_df.orphan_rows == 0).all()
    ),
    "All HEADER rows have GOVERNANCE": bool(
        (join_year.missing_governance == 0).all()
    ),
    "Valid EINs": bool(
        (key_audit.invalid_eins == 0).all()
    ),
    "Valid tax years": bool(
        (key_audit.invalid_years == 0).all()
    ),
}

structural_df = pd.DataFrame([
    {"check": name, "passed": passed}
    for name, passed in structural_checks.items()
])

print("STRUCTURAL INTEGRITY")
display(structural_df)

print(
    f"Checks passed: {structural_df.passed.sum()} "
    f"/ {len(structural_df)}"
)

structural_df.to_csv(
    out / "08_structural_integrity.csv",
    index=False
)


# 2. Data completeness
completeness_summary = con.execute("""
    SELECT
        COUNT(*) AS total_filings,

        COUNT(*) FILTER (
            WHERE summary_objectid IS NULL
        ) AS missing_summary,

        COUNT(*) FILTER (
            WHERE governance_objectid IS NULL
        ) AS missing_governance,

        COUNT(*) FILTER (
            WHERE raw_disclosure IS NULL
               OR TRIM(raw_disclosure) = ''
        ) AS missing_disclosure

    FROM master_audit
""").df()

for col in [
    "missing_summary",
    "missing_governance",
    "missing_disclosure"
]:
    completeness_summary[col + "_pct"] = (
        100 * completeness_summary[col]
        / completeness_summary["total_filings"]
    ).round(2)

print("\nDATA COMPLETENESS")
display(completeness_summary)

completeness_summary.to_csv(
    out / "08_data_completeness.csv",
    index=False
)


# 3. Duplicate EIN/year disclosure conflicts
duplicate_label_audit = con.execute("""
    WITH normalized AS (
        SELECT
            ein,
            tax_year,
            CASE
                WHEN UPPER(TRIM(raw_disclosure))
                    IN ('1', 'Y', 'YES', 'TRUE')
                THEN 1
                WHEN UPPER(TRIM(raw_disclosure))
                    IN ('0', 'N', 'NO', 'FALSE')
                THEN 0
                ELSE NULL
            END AS disclosure_label
        FROM master_audit
        WHERE ein IS NOT NULL
          AND tax_year IS NOT NULL
    ),
    duplicate_groups AS (
        SELECT
            ein,
            tax_year,
            COUNT(*) AS filings,
            COUNT(DISTINCT disclosure_label)
                AS distinct_known_labels,
            COUNT(*) FILTER (
                WHERE disclosure_label IS NULL
            ) AS unknown_labels,
            COUNT(*) FILTER (
                WHERE disclosure_label IS NOT NULL
            ) AS known_labels
        FROM normalized
        GROUP BY ein, tax_year
        HAVING COUNT(*) > 1
    )
    SELECT
        COUNT(*) AS duplicate_groups,
        COUNT(*) FILTER (
            WHERE distinct_known_labels > 1
        ) AS conflicting_known_labels,
        COUNT(*) FILTER (
            WHERE known_labels > 0
              AND unknown_labels > 0
        ) AS mixed_known_unknown,
        COUNT(*) FILTER (
            WHERE known_labels = 0
        ) AS all_labels_unknown
    FROM duplicate_groups
""").df()

print("\nDUPLICATE DISCLOSURE LABEL AUDIT")
display(duplicate_label_audit)

duplicate_label_audit.to_csv(
    out / "08_duplicate_label_conflicts.csv",
    index=False
)

print("\nAUDIT DECISION")
if structural_df.passed.all():
    print("Structural integrity: PASS")
else:
    print("Structural integrity: REVIEW REQUIRED")

print(
    "Data completeness: Known limitations documented."
)
print(
    "Duplicate disclosure conflicts: Review results above."
)
print(
    "Audit reports saved to:", out.resolve()
)

In [ ]:
columns = con.execute("""
    DESCRIBE SELECT * FROM header
""").df()

display(
    columns[
        columns["column_name"].str.contains(
            "EIN|ORG|NAME|ID",
            case=False,
            na=False
        )
    ]
)

In [ ]:
display(
    con.execute("""
        SELECT EIN2, COUNT(*) AS records
        FROM header
        GROUP BY EIN2
        ORDER BY records DESC
        LIMIT 10
    """).df()
)

In [ ]:
display(con.execute("""
    SELECT
        EIN2,
        ORG_EIN,
        F9_00_ORG_EIN,
        REGEXP_REPLACE(EIN2, '[^0-9]', '', 'g') AS ein2_clean
    FROM header
    WHERE EIN2 IS NOT NULL
    LIMIT 15
""").df())

In [ ]:
display(con.execute("""
    SELECT
        COUNT(*) AS total_records,
        COUNT(DISTINCT EIN2) AS distinct_ein2,
        COUNT(DISTINCT ORG_EIN) AS distinct_org_ein,
        COUNT(*) FILTER (
            WHERE REGEXP_REPLACE(EIN2, '[^0-9]', '', 'g')
                  = REGEXP_REPLACE(ORG_EIN, '[^0-9]', '', 'g')
        ) AS matching_eins,
        COUNT(*) FILTER (
            WHERE ORG_EIN IS NOT NULL
              AND REGEXP_REPLACE(EIN2, '[^0-9]', '', 'g')
                  <> REGEXP_REPLACE(ORG_EIN, '[^0-9]', '', 'g')
        ) AS mismatched_eins
    FROM header
""").df())

In [ ]:
conflict_review = con.execute("""
    WITH filings AS (
        SELECT
            m.ein,
            m.tax_year,
            m.OBJECTID,
            h.RETURN_TIME_STAMP,
            h.RETURN_AMENDED_X,
            CASE
                WHEN UPPER(TRIM(m.raw_disclosure))
                    IN ('1', 'Y', 'YES', 'TRUE') THEN 1
                WHEN UPPER(TRIM(m.raw_disclosure))
                    IN ('0', 'N', 'NO', 'FALSE') THEN 0
                ELSE NULL
            END AS disclosure_label
        FROM master_audit m
        JOIN header h
            ON m.OBJECTID = h.OBJECTID
    ),
    ranked AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY ein, tax_year
                ORDER BY
                    TRY_CAST(RETURN_TIME_STAMP AS TIMESTAMPTZ)
                        DESC NULLS LAST,
                    CASE
                        WHEN UPPER(TRIM(RETURN_AMENDED_X)) = 'TRUE'
                        THEN 1 ELSE 0
                    END DESC,
                    OBJECTID DESC
            ) AS filing_rank
        FROM filings
    ),
    groups AS (
        SELECT
            ein,
            tax_year,
            COUNT(*) AS filing_count,
            COUNT(DISTINCT disclosure_label) AS distinct_known_labels,
            COUNT(*) FILTER (
                WHERE disclosure_label IS NULL
            ) AS unknown_count,
            COUNT(*) FILTER (
                WHERE disclosure_label IS NOT NULL
            ) AS known_count,
            MAX(disclosure_label) FILTER (
                WHERE filing_rank = 1
            ) AS selected_label
        FROM ranked
        GROUP BY ein, tax_year
        HAVING COUNT(*) > 1
    )
    SELECT
        COUNT(*) FILTER (
            WHERE distinct_known_labels > 1
        ) AS conflicting_groups,
        COUNT(*) FILTER (
            WHERE distinct_known_labels > 1
              AND selected_label = 1
        ) AS conflicts_selected_positive,
        COUNT(*) FILTER (
            WHERE distinct_known_labels > 1
              AND selected_label = 0
        ) AS conflicts_selected_negative,
        COUNT(*) FILTER (
            WHERE distinct_known_labels > 1
              AND selected_label IS NULL
        ) AS conflicts_selected_unknown,
        COUNT(*) FILTER (
            WHERE known_count > 0
              AND unknown_count > 0
              AND selected_label IS NULL
        ) AS mixed_groups_selected_unknown
    FROM groups
""").df()

display(conflict_review)
conflict_review.to_csv(
    out / "08_duplicate_label_selection_impact.csv",
    index=False
)

### Duplicate disclosure selection findings

The audit identified 67 EIN/tax-year groups with conflicting known diversion-disclosure labels. Under the latest-filing selection rule, 53 were retained as positive disclosures and 14 as negative disclosures; none were retained with an unknown label.

Additionally, 512 EIN/tax-year groups containing both known and unknown disclosures were assigned an unknown label by the latest-filing rule.

These findings do not indicate a structural join failure. They represent source-level inconsistencies and label-selection uncertainty. The selected filings are retained for reproducibility, while conflicting groups should be flagged for sensitivity analysis in downstream modeling.

Unknown disclosure labels must remain NULL and must not be automatically converted to negative values or replaced with earlier known labels.